# Assignment 2 — Reproducing the Iqra'Eval 2025 benchmark baselines

**Paper.** El Kheir, Y., Ibrahim, O., Meghanani, A. et al. *Towards a Unified Benchmark for Arabic Pronunciation
Assessment: Qur'anic Recitation as Case Study.* Interspeech 2025, pp. 2410–2414. arXiv:2506.07722.
Code: https://github.com/Iqra-Eval/interspeech_IqraEval

**What this notebook does.** It evaluates the authors' **released checkpoints** (wav2vec2, WavLM, HuBERT, mHuBERT;
SSL upstream + weighted layer sum + 2-layer BiLSTM + CTC) with the authors' **unmodified** inference script
(`s3prl_inference.py`) and **official scorer** (`mdd_eval/`), and compares with the paper's QuranMB results
(training data CMV-Ar+TTS). No model is trained or tuned. Each checkpoint runs the released script twice (seeds 1 and 2: it leaves dropout on, so its output is random) and once more with `.eval()` as a clearly labelled, deterministic extra.

| Model | PER | Precision | Recall | F1 |
| --- | --- | --- | --- | --- |
| wav2vec2 | 19.47 | 16.16 | 77.07 | 26.72 |
| WavLM | 18.80 | 16.66 | 75.61 | 27.30 |
| HuBERT | 19.29 | 16.40 | 77.43 | 27.06 |
| mHuBERT | 16.42 | 18.70 | 74.29 | 29.88 |

*Paper setup (quoted, not re-run): frozen SSL upstream (weighted layer sum), 2-layer 1024-unit BiLSTM + CTC, greedy
decoding, Adam lr 1e-4, batch 16, 12.5k updates, early stopping on dev PER.*

**Run settings (Kaggle):** Accelerator **GPU T4** (one GPU is used), Internet **On**, secret **`HF_TOKEN`** (with access
to the gated `IqraEval/IqraEval_Test_GT`). No datasets need attaching. Expected time: ~60–70 min (environment ~10 min; 3 inference passes per checkpoint).
Every deviation from the paper or README is recorded in `DEVIATIONS` the moment it is found and printed in the
**Deviations** cell at the end.


In [ ]:
# ---- 0. Settings, deviation log, environment record ---------------------------------------------------------
import os, sys, json, time, shutil, subprocess, hashlib, platform, re, ast, io
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path(os.environ.get("A2_BASE", "/kaggle/working"))      # outputs (saved with the notebook)
TMP = Path(os.environ.get("A2_TMP", "/tmp/a2"))                  # environments, sources, audio (not saved)
LIMIT = int(os.environ["A2_LIMIT"]) if os.environ.get("A2_LIMIT") else None   # None = the full test set
OUT = BASE / "outputs"
for p in (OUT, TMP):
    p.mkdir(parents=True, exist_ok=True)

SCORER_REPO, SCORER_COMMIT = "https://github.com/Iqra-Eval/interspeech_IqraEval.git", "12ebde7829ad6a97a0f131457c2efb7fc095b257"
S3PRL_REPO, S3PRL_COMMIT = "https://github.com/s3prl/s3prl.git", "ec8064b5889f81ca460fbe2c094ce576a6f120b7"
CKPT_URL = "https://huggingface.co/Trikaldarshi/sws_pretrained_models/resolve/main/"
MODELS = {"wav2vec2": "wave2vec2_base.ckpt", "wavlm": "wavlm_base.ckpt", "hubert": "hubert_base.ckpt", "mhubert": "mhubert.ckpt"}
PAPER = {"wav2vec2": (19.47, 16.16, 77.07, 26.72), "wavlm": (18.80, 16.66, 75.61, 27.30),
         "hubert": (19.29, 16.40, 77.43, 27.06), "mhubert": (16.42, 18.70, 74.29, 29.88)}   # PER, P, R, F1 (%)

DEVIATIONS = []
def deviation(topic, text):
    DEVIATIONS.append({"n": len(DEVIATIONS) + 1, "topic": topic, "deviation": text})
    print(f"  >> DEVIATION {len(DEVIATIONS)} [{topic}]: {text}")

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, encoding="utf-8", errors="replace", **kw)
    if r.returncode:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}\n{r.stdout[-2000:]}\n{r.stderr[-3000:]}")
    return r.stdout.strip()

def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
assert os.environ.get("HF_TOKEN"), "HF_TOKEN is required (IqraEval_Test_GT is gated)"
ENV = {"notebook_python": sys.version.split()[0], "platform": platform.platform(), "limit": LIMIT}
try:
    ENV["gpu"] = sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader")
except Exception:
    ENV["gpu"] = "none"
print(json.dumps(ENV, indent=1))
if LIMIT:
    print(f"*** LIMIT={LIMIT}: plumbing test only, NOT the reproduction ***")


In [ ]:
# ---- S1a. Sources, pinned: the authors' repo (inference script + scorer) and s3prl ------------------------------
SRC = TMP / "src"
SRC.mkdir(exist_ok=True)
IQ, S3 = SRC / "interspeech_IqraEval", SRC / "s3prl"

def fetch_commit(url, commit, dst, depth=None, tries=4):
    # Fetch exactly one pinned commit (shallow for s3prl, a large repo), with retries for flaky connections
    for t in range(1, tries + 1):
        try:
            if not (dst / ".git").exists():
                dst.mkdir(parents=True, exist_ok=True)
                sh(["git", "-C", str(dst), "init", "-q"])
                sh(["git", "-C", str(dst), "remote", "add", "origin", url])
            sh(["git", "-C", str(dst), "fetch", "-q"] + (["--depth", str(depth)] if depth else []) + ["origin", commit])
            sh(["git", "-C", str(dst), "checkout", "-q", "FETCH_HEAD"])
            return
        except RuntimeError as e:
            print(f"  fetch {url} attempt {t} failed: {str(e).splitlines()[-1][:150]}")
            time.sleep(5 * t)
    raise RuntimeError(f"could not fetch {url} @ {commit}")

fetch_commit(SCORER_REPO, SCORER_COMMIT, IQ)                  # full history: needed for the mdd_eval/ log below
fetch_commit(S3PRL_REPO, S3PRL_COMMIT, S3, depth=1)
ENV["interspeech_IqraEval_commit"] = sh(["git", "-C", str(IQ), "rev-parse", "HEAD"])
ENV["s3prl_commit"] = sh(["git", "-C", str(S3), "rev-parse", "HEAD"])
ENV["s3prl_commit_date"] = sh(["git", "-C", str(S3), "log", "-1", "--format=%ad", "--date=short"])
print("interspeech_IqraEval:", ENV["interspeech_IqraEval_commit"], "| s3prl:", ENV["s3prl_commit"], ENV["s3prl_commit_date"])
ENV["sha256_s3prl_inference.py"] = sha256(IQ / "s3prl_inference.py")
for f in ("align_data.py", "ins_del_cor_sub_analysis.py", "metric.py"):
    ENV[f"sha256_mdd_eval/{f}"] = sha256(IQ / "mdd_eval" / f)

# Did the scorer change after the paper (June 2025)?
log = sh(["git", "-C", str(IQ), "log", "--all", "--format=%h %ad %s", "--date=short", "--", "mdd_eval/"])
print("mdd_eval history:\n" + log)
dates = [l.split()[1] for l in log.splitlines()]
ENV["mdd_eval_history"] = log.splitlines()
if all(d < "2025-06-01" for d in dates):
    print("scorer unchanged since before the paper: one scoring pass with the pinned commit")
else:
    deviation("scorer", f"mdd_eval/ changed after June 2025: {log}")


In [ ]:
# ---- S1b. Python 3.11 side environment for inference (s3prl needs torchaudio < 2.9) --------------------------------
PY = TMP / "py311" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not PY.exists():
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
    sh([sys.executable, "-m", "uv", "python", "install", "3.11"])
    sh([sys.executable, "-m", "uv", "venv", "-q", "-p", "3.11", str(TMP / "py311")])
    idx = "https://download.pytorch.org/whl/cu121" if ENV["gpu"] != "none" else "https://download.pytorch.org/whl/cpu"
    sh([sys.executable, "-m", "uv", "pip", "install", "-q", "-p", str(PY), "torch==2.4.1", "torchaudio==2.4.1", "--index-url", idx])
    sh([sys.executable, "-m", "uv", "pip", "install", "-q", "-p", str(PY), "-e", str(S3), "editdistance", "tensorboardX",
        "huggingface_hub<1.0", "transformers==4.44.2", "pandas", "soundfile", "requests"])
ver = json.loads(sh([str(PY), "-c", "import json, sys, torch, torchaudio, transformers, huggingface_hub; "
                     "from importlib.metadata import version; print(json.dumps({'python': sys.version.split()[0], "
                     "'torch': torch.__version__, 'torchaudio': torchaudio.__version__, 's3prl': version('s3prl'), "
                     "'transformers': transformers.__version__, 'huggingface_hub': huggingface_hub.__version__, "
                     "'cuda': torch.cuda.is_available(), 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}))"]))
ENV["inference_env"] = ver
print("inference environment:", ver)
deviation("environment", f"README asks for Python 3.8 (conda); Kaggle runs Python {ENV['notebook_python']}. s3prl imports every "
          "upstream at load time and several use torchaudio APIs removed in torchaudio>=2.9 (set_audio_backend, sox_effects), "
          f"so inference runs in a uv-created side environment: Python {ver['python']}, torch {ver['torch']}, torchaudio "
          f"{ver['torchaudio']}, s3prl {ver['s3prl']} (commit {ENV['s3prl_commit'][:7]}, {ENV['s3prl_commit_date']}), "
          f"transformers {ver['transformers']}, huggingface_hub {ver['huggingface_hub']}. No code shims were needed there.")


In [ ]:
# ---- S1b'. Wrapper around the authors' script: runs s3prl_inference.py unmodified (runpy); shims only if needed ----
TOOLS = TMP / "tools"                    # kept apart from the s3prl clone so it cannot shadow the package
TOOLS.mkdir(exist_ok=True)
WRAPPER_SRC = r'''"""Wrapper around the authors' s3prl_inference.py (run unmodified via runpy).

Compatibility shims, applied before the script imports s3prl; each is a deviation logged in the notebook:
  1. huggingface_hub >= 1.0 removed `Repository` and `HfFolder`; s3prl.downstream.runner imports them at module
     level but uses them only to upload models to the Hub. Placeholders are added only if they are missing.
  2. torchaudio 2.x removed set_audio_backend; s3prl imports every upstream at load time and one unrelated
     upstream (byol_s) calls it. A no-op is added only if it is missing.
  3. torchaudio >= 2.9 routes torchaudio.load through torchcodec. If torchaudio.load cannot read a WAV here, it is
     replaced by a soundfile reader with the same contract: float32 tensor [channels, samples] in [-1, 1], native
     sample rate (no resampling), exactly what the script expects.
Options added by the wrapper (not passed to the script):
  --seed N  seeds torch/numpy/random before the script runs (the released script runs with dropout on, so its
            predictions are random; a seed makes each run repeatable).
  --eval    calls .eval() on the upstream, featurizer and downstream as the script builds them (the s3prl runner's
            evaluation path does this; the released script does not). Clearly labelled extra, deterministic.
The mode each part was in at inference is printed as MODES.
Usage: python run_s3prl_inference.py <path to s3prl_inference.py> [--seed N] [--eval] --ckpt ... --dict_path ... --wav_dir ... --output_csv ...
"""
import json
import runpy
import sys

SHIMS = []

import huggingface_hub  # noqa: E402

for _name in ("Repository", "HfFolder"):
    if not hasattr(huggingface_hub, _name):
        setattr(huggingface_hub, _name, type(_name, (), {"__doc__": "placeholder: removed in huggingface_hub>=1.0"}))
        SHIMS.append(f"huggingface_hub.{_name} placeholder")

import numpy as np  # noqa: E402
import soundfile as sf  # noqa: E402
import torch  # noqa: E402
import torchaudio  # noqa: E402


if not hasattr(torchaudio, "set_audio_backend"):           # removed in torchaudio 2.x; s3prl's byol_s upstream
    torchaudio.set_audio_backend = lambda *a, **k: None     # (unused here) calls it at import time
    SHIMS.append("torchaudio.set_audio_backend no-op")


def _sf_load(path, *args, **kwargs):
    data, sr = sf.read(str(path), dtype="float32", always_2d=True)      # [samples, channels]
    return torch.from_numpy(np.ascontiguousarray(data.T)), sr


def _probe(path):
    try:
        wav, sr = torchaudio.load(path)
        ref, rsr = _sf_load(path)
        return wav.shape == ref.shape and sr == rsr and torch.allclose(wav, ref, atol=1e-4)
    except Exception:
        return False


def _watch_modes(force_eval):
    """Record whether upstream / featurizer / downstream are in training mode when the script builds them
    (the released script never calls .eval(), so dropout stays on). With force_eval, call .eval() on each first
    (the s3prl runner's own evaluation path does this); nothing else changes."""
    from s3prl.downstream.runner import Runner
    modes = {}
    for part in ("upstream", "featurizer", "downstream"):
        orig = getattr(Runner, f"_get_{part}")

        def patched(self, _orig=orig, _part=part):
            entry = _orig(self)
            if force_eval:
                entry.model.eval()
            modes[_part] = "train (dropout on)" if entry.model.training else "eval"
            return entry
        setattr(Runner, f"_get_{part}", patched)
    return modes


if __name__ == "__main__":
    # wrapper options (removed before the script sees argv): --seed N (torch/numpy/random seed), --eval
    args = sys.argv[1:]
    seed, force_eval = None, False
    if "--seed" in args:
        i = args.index("--seed"); seed = int(args[i + 1]); del args[i:i + 2]
    if "--eval" in args:
        args.remove("--eval"); force_eval = True
    script, argv = args[0], args[1:]
    if seed is not None:
        import random
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    MODES = _watch_modes(force_eval)
    import atexit
    atexit.register(lambda: print("MODES:", json.dumps({"seed": seed, "eval": force_eval, **MODES}), flush=True))
    wav_dir = argv[argv.index("--wav_dir") + 1]
    from pathlib import Path
    first = next(iter(sorted(Path(wav_dir).glob("*.wav"))), None)
    if first is not None and not _probe(str(first)):
        torchaudio.load = _sf_load
        SHIMS.append("torchaudio.load -> soundfile reader (torchaudio could not read WAV here)")
    print("SHIMS:", json.dumps(SHIMS), flush=True)
    sys.argv = [script] + argv
    runpy.run_path(script, run_name="__main__")
'''
(TOOLS / "run_s3prl_inference.py").write_text(WRAPPER_SRC, encoding="utf-8")
print("wrapper written:", TOOLS / "run_s3prl_inference.py")


In [ ]:
# ---- S1c. Released checkpoints: download, SHA-256 (vs Hugging Face's own record), contents ----------------------
from huggingface_hub import HfApi
api = HfApi()
lfs = {s.rfilename: (s.lfs or {}) for s in api.model_info("Trikaldarshi/sws_pretrained_models", files_metadata=True).siblings}
import tempfile
# s3prl_inference.py deletes its --ckpt and --dict_path files after a run when they lie under the system temp directory
# (it assumes it downloaded them from a URL). So the checkpoints live outside it.
CK = Path(os.environ.get("A2_CKPT", str(Path.home() / "a2_ckpt")))
assert not str(CK.resolve()).startswith(str(Path(tempfile.gettempdir()).resolve())), CK
CK.mkdir(parents=True, exist_ok=True)
HASHES = {}
for name, fn in list(MODELS.items()) + [("dict", "sws_arabic.txt")]:
    dst = CK / fn
    if not dst.exists():
        sh(["curl", "-sSL", "--fail", "--retry", "4", "--retry-all-errors", "-o", str(dst), CKPT_URL + fn])
    HASHES[fn] = sha256(dst)
    meta = lfs.get(fn)
    hf = (meta.get("sha256") if isinstance(meta, dict) else getattr(meta, "sha256", None)) if meta else None
    if hf and HASHES[fn] != hf:                  # partial/corrupt download: fetch once more, then it must match
        print(f"{fn}: hash mismatch ({dst.stat().st_size:,d} B), downloading again")
        dst.unlink()
        sh(["curl", "-sSL", "--fail", "--retry", "4", "--retry-all-errors", "-o", str(dst), CKPT_URL + fn])
        HASHES[fn] = sha256(dst)
    print(f"{fn:22s} {dst.stat().st_size:>11,d} B  sha256 {HASHES[fn]}" + (f"  (HF: {'match' if hf == HASHES[fn] else 'MISMATCH ' + str(hf)})" if hf else ""))
    if hf:
        assert hf == HASHES[fn], fn
deviation("inference script", "s3prl_inference.py's cleanup() deletes the --ckpt and --dict_path files after each run when "
          "their path starts with the system temp directory (meant for files it downloaded from a URL). Checkpoints and the "
          f"dictionary are therefore kept in {CK}, outside {tempfile.gettempdir()}; the script itself is unchanged.")
deviation("checkpoint name", "the wav2vec2 checkpoint is published as 'wave2vec2_base.ckpt' (the spec/README name "
          "'wav2vec2_base.ckpt' does not exist); used as published")
(OUT / "checkpoint_sha256.json").write_text(json.dumps(HASHES, indent=1))

INSPECT = r'''
import torch, argparse, json, sys
torch.serialization.add_safe_globals([argparse.Namespace])
d = torch.load(sys.argv[1], map_location="cpu", weights_only=False)
a, c = d["Args"], d["Config"]
n = lambda v: sum(t.numel() for t in v.values() if hasattr(t, "numel")) if isinstance(v, dict) else None
print(json.dumps({"keys": list(d), "step": d.get("Step"), "epoch": d.get("Epoch"), "upstream": a.upstream,
                  "upstream_ckpt": getattr(a, "upstream_ckpt", None), "upstream_feature": getattr(a, "upstream_feature_selection", None),
                  "upstream_trainable": getattr(a, "upstream_trainable", None), "downstream": a.downstream,
                  "total_steps": c["runner"].get("total_steps"), "eval_step": c["runner"].get("eval_step"),
                  "lr": c.get("optimizer", {}).get("lr"), "downstream_params": n(d.get("Downstream")),
                  "featurizer_params": n(d.get("Featurizer"))}, default=str))
'''
CKINFO = {name: json.loads(sh([str(PY), "-c", INSPECT, str(CK / fn)]).splitlines()[-1]) for name, fn in MODELS.items()}
print(pd.DataFrame(CKINFO).T[["upstream", "upstream_ckpt", "step", "epoch", "total_steps", "eval_step", "lr", "downstream_params"]].to_string())
steps = {k: v["step"] for k, v in CKINFO.items()}
deviation("training steps (unconfirmed)", f"the checkpoints store Step={steps}, while the paper reports 12.5k updates. The repo's "
          f"config/sws.yaml sets runner.total_steps={CKINFO['mhubert']['total_steps']} and eval_step={CKINFO['mhubert']['eval_step']}, "
          "so the stored value is probably the configured total or the step the file was written at, not the number of updates "
          "used. Both numbers are recorded; the explanation is unconfirmed.")


In [ ]:
# ---- S2. Test audio (QuranMB.v2 test split) and the answer files ------------------------------------------------------
import soundfile as sf
from datasets import load_dataset, Audio
qmb = load_dataset("IqraEval/QuranMB.v2", split="test").cast_column("audio", Audio(decode=False))
WAV = TMP / "wav_test"
WAV.mkdir(exist_ok=True)
ids, secs, rates, chans, rewritten = [], 0.0, set(), set(), 0
for k, ex in enumerate(qmb):
    if LIMIT and k >= LIMIT:
        break
    cid, raw = str(ex["ID"]), ex["audio"]["bytes"]
    info = sf.info(io.BytesIO(raw))
    rates.add(info.samplerate); chans.add(info.channels)
    if info.samplerate == 16000 and info.channels == 1 and info.format == "WAV":
        (WAV / f"{cid}.wav").write_bytes(raw)             # the original file, byte for byte
    else:                                                   # resample/downmix only if needed (logged)
        a, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
        a = a.mean(axis=1)
        if sr != 16000:
            from scipy.signal import resample_poly
            g = np.gcd(sr, 16000); a = resample_poly(a, 16000 // g, sr // g)
        sf.write(WAV / f"{cid}.wav", a, 16000, subtype="PCM_16"); rewritten += 1
    ids.append(cid); secs += info.duration
print(f"test clips {len(ids)} | {secs / 3600:.2f} h | sample rates {sorted(rates)} | channels {sorted(chans)} | re-encoded {rewritten}")
ENV["test"] = {"clips": len(ids), "hours": round(secs / 3600, 3), "sample_rates": sorted(rates), "re_encoded": rewritten}
if rewritten == 0:
    print("all clips already 16 kHz mono WAV: written unchanged (the spec's resampling step is a no-op)")
else:
    deviation("test audio", f"{rewritten} of {len(ids)} clips were not 16 kHz mono WAV (sample rates {sorted(rates)}, channels "
              f"{sorted(chans)}); they were downmixed to mono (mean of the channels), resampled only if not 16 kHz, and written "
              "as 16-bit PCM WAV. The authors' script averages the channels itself (wav.mean(0)), so the only difference from "
              f"feeding the original files is 16-bit rounding of that mean. The other {len(ids) - rewritten} clips were written "
              "byte for byte.")

# The answer files: current version and any older revision (the paper predates the 2026-01-15 update)
REPO_GT = "IqraEval/IqraEval_Test_GT"
commits = api.list_repo_commits(REPO_GT, repo_type="dataset", token=os.environ["HF_TOKEN"])
hist = [{"commit": c.commit_id, "date": str(c.created_at)[:19], "title": c.title} for c in commits]
print(pd.DataFrame(hist).to_string(index=False))
ENV["answer_file_history"] = hist

def truth_at(rev):
    files = api.list_repo_files(REPO_GT, repo_type="dataset", revision=rev, token=os.environ["HF_TOKEN"])
    csvs = [f for f in files if f.endswith(".csv")]
    if csvs:
        from huggingface_hub import hf_hub_download
        df = pd.read_csv(hf_hub_download(REPO_GT, csvs[0], repo_type="dataset", revision=rev, token=os.environ["HF_TOKEN"]),
                         keep_default_na=False, dtype=str)
    else:
        df = load_dataset(REPO_GT, split="test", revision=rev).to_pandas().astype(str)
    ren = {c: "Annotation_phn" for c in df.columns if c in ("Annotated_phn", "annotation_phn")}
    return df.rename(columns=ren)[["ID", "Reference_phn", "Annotation_phn"]], files, ren

TRUTH, TRUTH_INFO = {}, {}
cur = hist[0]["commit"]
TRUTH["current"], files, ren = truth_at(cur)
TRUTH_INFO["current"] = {"commit": cur, "date": hist[0]["date"], "files": files, "renamed": ren}
older = [h for h in hist[1:] if h["date"] < "2025-06-09"] or [h for h in hist[1:] if h["date"] < "2026-01-15"]
old_ok = False
for h in older:                                     # newest qualifying older revision with a usable answer file
    try:
        TRUTH["old"], files, ren = truth_at(h["commit"])
        TRUTH_INFO["old"] = {"commit": h["commit"], "date": h["date"], "files": files, "renamed": ren}
        old_ok = True
        break
    except Exception as e:
        print("  revision", h["commit"][:8], h["date"], "not usable:", repr(e)[:200])
if old_ok:
    o, c = TRUTH["old"].set_index("ID"), TRUTH["current"].set_index("ID")
    common = o.index.intersection(c.index)
    diff_ann = int((o.loc[common, "Annotation_phn"] != c.loc[common, "Annotation_phn"]).sum())
    diff_ref = int((o.loc[common, "Reference_phn"] != c.loc[common, "Reference_phn"]).sum())
    TRUTH_INFO["old_vs_current"] = {"old_clips": len(o), "current_clips": len(c), "common": len(common),
                                    "annotation_differs": diff_ann, "reference_differs": diff_ref}
    print("older answer file found:", TRUTH_INFO["old"]["commit"][:8], TRUTH_INFO["old"]["date"], TRUTH_INFO["old_vs_current"])
else:
    deviation("answer file", "no revision of IqraEval/IqraEval_Test_GT older than the current one is accessible, so the "
              "paper's exact labels cannot be used; both tables are scored against the current labels "
              f"(commit {cur[:8]}, {hist[0]['date']})")
if ren:
    deviation("answer file", f"column renamed for the scorer: {ren}")
SETS = {"v1": "old" if old_ok else "current", "v2": "current"}
for s, key in SETS.items():
    t = TRUTH[key][TRUTH[key]["ID"].isin(ids)].reset_index(drop=True)
    assert set(t["ID"]) == set(ids), (s, len(t), len(ids))
    t.to_csv(OUT / f"truth_{s}.csv", index=False)
    print(f"truth_{s}.csv: {len(t)} rows (labels: {key})")

n_sent = TRUTH["current"]["Reference_phn"].nunique()
print(f"size check: {len(ids)} clips, {n_sent} distinct canonical sentences, {secs / 3600:.2f} h "
      "(paper: 98 verses, 18 speakers, ~2.2 h)")
deviation("test set", "QuranMB.v1 is the same audio as QuranMB.v2 (IqraEval/open_testset redirects to QuranMB.v2; the audio "
          "file is unchanged since 2025-07-05), so 'v1' uses the full test audio, not a subset. Size as found: "
          f"{len(ids)} clips, {n_sent} distinct canonical sentences, {secs / 3600:.2f} h vs the paper's 98 verses, "
          "18 speakers, ~2.2 h (speakers are not listed in the data)."
          + (f" Older answer file: {TRUTH_INFO['old_vs_current']['old_clips']} clips." if old_ok else ""))


In [ ]:
# ---- S3a. Pin and prefetch every SSL upstream (before any inference) ---------------------------------------------------
# The released checkpoints do not contain the SSL upstream; s3prl fetches it at load time without a revision and without
# verifying the file (on Kaggle a dropped connection left a truncated wav2vec_small.pt that s3prl then tried to load).
# So each upstream is resolved to a commit, downloaded here at exactly that commit, checked against Hugging Face's
# SHA-256, and placed where s3prl looks; s3prl then uses these files and downloads nothing.
from huggingface_hub import hf_hub_download
S3PRL_HF = "s3prl/converted_ckpts"            # where s3prl (legacy=False) downloads its built-in upstreams from
S3PRL_FILES = {"wav2vec2_base_960": "wav2vec_small.pt", "hubert_base": "hubert_base_ls960.pt", "wavlm_base": "wavlm_base.pt"}
S3PRL_DIR = Path.home() / ".cache" / "s3prl" / "download"
S3PRL_DIR.mkdir(parents=True, exist_ok=True)

def hf_file_sha(repo, fn, rev):
    for s in api.model_info(repo, revision=rev, files_metadata=True).siblings:
        if s.rfilename == fn:
            m = s.lfs
            return (m.get("sha256") if isinstance(m, dict) else getattr(m, "sha256", None)) if m else None

PIN_SRC = r'''
import json, sys
from huggingface_hub import HfApi, snapshot_download
repo = sys.argv[1]
sha = HfApi().model_info(repo).sha
for t in range(4):
    try:
        path = snapshot_download(repo, revision=sha)
        snapshot_download(repo)                    # refs/main -> the commit just logged (used offline below)
        break
    except Exception as e:
        if t == 3: raise
print(json.dumps({"repo": repo, "commit": sha, "path": path}))
'''
UPSTREAMS = {}
s3prl_commit = api.model_info(S3PRL_HF).sha
for name, info in CKINFO.items():
    up, src = info["upstream"], info["upstream_ckpt"]
    if up in S3PRL_FILES:
        fn = S3PRL_FILES[up]
        url = f"https://huggingface.co/{S3PRL_HF}/resolve/main/{fn}"
        dest = S3PRL_DIR / f"{hashlib.sha256(url.encode()).hexdigest()}.{fn}"      # s3prl's own cache naming
        want = hf_file_sha(S3PRL_HF, fn, s3prl_commit)
        for t in range(4):
            try:
                p = hf_hub_download(S3PRL_HF, fn, revision=s3prl_commit)
                if want is None or sha256(p) == want:
                    break
                os.remove(p)
            except Exception as e:
                print(f"  {fn}: attempt {t + 1} failed: {repr(e)[:150]}")
                time.sleep(5 * (t + 1))
        else:
            raise RuntimeError(f"could not download {S3PRL_HF}/{fn} at {s3prl_commit}")
        shutil.copyfile(p, dest)                     # overwrites any truncated file s3prl left behind
        got = sha256(dest)
        assert want is None or got == want, (fn, got, want)
        UPSTREAMS[name] = {"type": "s3prl upstream (Hugging Face file)", "upstream": up, "repo": S3PRL_HF, "file": fn,
                           "commit": s3prl_commit, "sha256": got, "s3prl_cache_path": str(dest)}
    elif src and re.fullmatch(r"[\w.-]+/[\w.-]+", str(src)):
        UPSTREAMS[name] = {"type": "Hugging Face model", "upstream": up,
                           **json.loads(sh([str(PY), "-c", PIN_SRC, src]).splitlines()[-1])}
    else:
        raise RuntimeError(f"{name}: unknown upstream {up!r} / {src!r}: add how to pin it")
for k, v in UPSTREAMS.items():
    print(f"{k:9s} {v['repo']}{'/' + v['file'] if 'file' in v else ''} @ {v['commit']}" + (f" | sha256 {v['sha256'][:16]}" if 'sha256' in v else ""))
(OUT / "upstream_pins.json").write_text(json.dumps(UPSTREAMS, indent=1))
deviation("upstream weights", "the released checkpoints do not contain the SSL upstream; s3prl loads it at run time without "
          "a revision. Pinned here instead: each upstream was downloaded at a resolved commit (SHA-256 checked) and placed in "
          "s3prl's cache, and inference ran offline on those files: " + "; ".join(
              f"{k}: {v['repo']}{'/' + v['file'] if 'file' in v else ''}@{v['commit'][:10]}" for k, v in UPSTREAMS.items()))


In [ ]:
# ---- S3b. 5-clip check on all four checkpoints (fails in a minute, not 30 minutes into the full run) ------------------
WRAP = TOOLS / "run_s3prl_inference.py"
SCRIPT = IQ / "s3prl_inference.py"
W5 = TMP / "wav5"
W5.mkdir(exist_ok=True)
for cid in ids[:5]:
    if not (W5 / f"{cid}.wav").exists():
        shutil.copy(WAV / f"{cid}.wav", W5 / f"{cid}.wav")

def infer(name, wav_dir, out_csv, extra=()):
    env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8", CUDA_VISIBLE_DEVICES="0", TF_CPP_MIN_LOG_LEVEL="3",
               HF_HUB_OFFLINE="1")               # only the pinned, cached upstream files are used
    t0 = time.time()
    r = subprocess.run([str(PY), str(WRAP), str(SCRIPT), *extra, "--ckpt", str(CK / MODELS[name]), "--dict_path",
                        str(CK / "sws_arabic.txt"), "--wav_dir", str(wav_dir), "--output_csv", str(out_csv)],
                       capture_output=True, text=True, encoding="utf-8", errors="replace", env=env, cwd=str(TOOLS))
    dt = time.time() - t0
    for f in (CK / MODELS[name], CK / "sws_arabic.txt"):
        assert f.exists(), f"{f} was deleted by the run"
    if r.returncode:
        raise RuntimeError(f"{name}: inference failed\n{r.stdout[-1500:]}\n{r.stderr[-4000:]}")
    if "Downloading:" in r.stdout + r.stderr:
        raise RuntimeError(f"{name}: s3prl downloaded something despite the prefetch:\n{(r.stdout + r.stderr)[-1500:]}")
    shims = re.search(r"SHIMS: (\[.*\])", r.stdout)
    modes = re.search(r"MODES: (\{.*\})", r.stdout)
    return dt, (json.loads(shims.group(1)) if shims else None), (json.loads(modes.group(1)) if modes else None), r.stdout

for name in MODELS:
    dt, shims, modes, _ = infer(name, W5, TMP / f"smoke_{name}.csv", ["--seed", "1"])
    df = pd.read_csv(TMP / f"smoke_{name}.csv")
    print(f"{name:9s} OK in {dt:.0f} s | shims {shims} | modes {modes} | e.g. {df['filename'].iloc[0]}: {df['output'].iloc[0][:60]}")
    if shims:
        deviation("compatibility shim", f"{name}: {shims}")


In [ ]:
# ---- S3c. Inference: the released script twice (seeds 1, 2), then once in eval mode (labelled extra) ------------------
# The released s3prl_inference.py never calls .eval(), so dropout (0.2 in the BiLSTM, plus the upstream's) stays on and its
# predictions are random. Main result: the script exactly as released, two seeded runs (mean and spread across seeds).
# Extra: the same script with .eval() on each part, as the s3prl runner's own evaluation does (deterministic).
def to_pred(raw_csv):
    df = pd.read_csv(raw_csv)
    rows = []
    for fn, out in zip(df["filename"], df["output"]):
        hyp = ast.literal_eval(out)                 # the script writes a stringified list of hypotheses
        assert isinstance(hyp, list) and len(hyp) == 1, (fn, out)
        rows.append({"ID": Path(fn).stem, "Prediction": " ".join(str(hyp[0]).split())})
    return pd.DataFrame(rows)

VARIANTS = {"released_seed1": ["--seed", "1"], "released_seed2": ["--seed", "2"], "eval": ["--eval"]}
TIMES, PRED, MODES = {}, {}, {}
for variant, extra in VARIANTS.items():
    for name in MODELS:
        raw = OUT / f"raw_{name}_{variant}.csv"
        dt, shims, modes, _ = infer(name, WAV, raw, extra)
        want = "eval" if variant == "eval" else "train (dropout on)"
        assert modes and all(modes[k] == want for k in ("upstream", "featurizer", "downstream")), (name, variant, modes)
        pred = to_pred(raw)
        assert len(pred) == len(ids) and set(pred["ID"]) == set(ids), (name, variant, len(pred))
        empty = int((pred["Prediction"].str.strip() == "").sum())
        assert empty == 0, f"{name} {variant}: {empty} empty predictions"
        PRED[(name, variant)], MODES[(name, variant)] = pred, modes
        TIMES[(name, variant)] = {"seconds": round(dt, 1), "sec_per_audio_hour": round(dt / (secs / 3600), 1), "clips": len(pred)}
        pred.to_csv(OUT / f"pred_{name}_{variant}.csv", index=False)
        print(f"{variant:15s} {name:9s} {dt / 60:4.1f} min ({TIMES[(name, variant)]['sec_per_audio_hour']:.0f} s per audio hour) "
              f"| modes {modes['upstream']}/{modes['featurizer']}/{modes['downstream']} | e.g. {pred['ID'].iloc[0]}: {pred['Prediction'].iloc[0][:60]}")
def _same_order(a, b):
    m = a.merge(b, on="ID", suffixes=("_1", "_2"))
    return int((m["Prediction_1"] != m["Prediction_2"]).sum())
changed = {name: _same_order(PRED[(name, "released_seed1")], PRED[(name, "released_seed2")]) for name in MODELS}
print("clips whose prediction differs between seed 1 and seed 2:", changed)
(OUT / "inference_modes.json").write_text(json.dumps({f"{k[0]}/{k[1]}": v for k, v in MODES.items()}, indent=1))


In [ ]:
# ---- S4. Official scoring (fresh folder per call) and PER ---------------------------------------------------------------
MDD = IQ / "mdd_eval"
def score(truth_csv, pred_csv, out_dir):
    out_dir = Path(out_dir)
    assert not out_dir.exists(), f"{out_dir} exists: the results file is opened in append mode, use a fresh folder"
    out_dir.mkdir(parents=True)
    aligned = out_dir / "aligned"
    env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1")
    r1 = subprocess.run([sys.executable, str(MDD / "align_data.py"), "--truth-file", str(truth_csv),
                         "--pred-file", str(pred_csv), "--output-dir", str(aligned)], capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
    r2 = subprocess.run([sys.executable, str(MDD / "ins_del_cor_sub_analysis.py"), "--align-folder", str(aligned) + "/"],
                        capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
    for r in (r1, r2):
        assert r.returncode == 0, r.stderr[-2000:]
    (out_dir / "scorer_stdout.txt").write_text(r1.stdout + "\n" + r2.stdout)
    get = lambda pat: float(re.search(pat, r2.stdout).group(1))
    return {"precision": get(r"Precision:\s*([0-9.]+)"), "recall": get(r"Recall:\s*([0-9.]+)"),
            "f1": get(r"f1-score:\s*([0-9.]+)")}

def edit_distance(a, b):
    d = np.arange(len(b) + 1)
    for i, x in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, y in enumerate(b, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (x != y))
    return int(d[len(b)])

def per(truth_df, pred_df):
    m = truth_df.merge(pred_df, on="ID")
    errs = sum(edit_distance(p.split(), a.split()) for p, a in zip(m["Prediction"], m["Annotation_phn"]))
    return errs / sum(len(a.split()) for a in m["Annotation_phn"])

print("PER = sum over clips of edit_distance(Prediction, Annotation_phn) / total Annotation_phn phonemes "
      "(recognition against what was actually said, as the paper's PER)")
LABELS = {}                                          # label version -> truth file (v1 and v2 share it if the labels are equal)
for s, key in SETS.items():
    LABELS.setdefault(key, OUT / f"truth_{s}.csv")
RES = []
for key, tfile in LABELS.items():
    truth = pd.read_csv(tfile, keep_default_na=False, dtype=str)
    for variant in VARIANTS:
        for name in MODELS:
            m = score(tfile, OUT / f"pred_{name}_{variant}.csv", OUT / f"score_{name}_{variant}_{key}")
            t = TIMES[(name, variant)]
            RES.append({"model": name, "variant": variant, "labels": key, "PER": round(100 * per(truth, PRED[(name, variant)]), 2),
                        "P": round(100 * m["precision"], 2), "R": round(100 * m["recall"], 2), "F1": round(100 * m["f1"], 2),
                        "clips": t["clips"], "inference_s": t["seconds"], "s_per_audio_hour": t["sec_per_audio_hour"]})
results = pd.DataFrame(RES)
results.to_csv(OUT / "results.csv", index=False)
print(results.to_string(index=False))


In [ ]:
# ---- S5. Comparison with the paper ----------------------------------------------------------------------------------------
M = ["PER", "P", "R", "F1"]
v1key = SETS["v1"]
v1lab = ("older labels (" + TRUTH_INFO["old"]["date"][:10] + ")") if old_ok else "current labels (2026-01-15)"
def row(name, variant, key=v1key):
    return results[(results["model"] == name) & (results["variant"] == variant) & (results["labels"] == key)].iloc[0]

print("MAIN RESULT: the released script exactly as published (dropout on), mean ± spread over seeds 1 and 2 "
      "(spread = half the difference between the two runs)")
SUMMARY = []
for name in MODELS:
    a, b, e = row(name, "released_seed1"), row(name, "released_seed2"), row(name, "eval")
    p = dict(zip(M, PAPER[name]))
    mean = {k: (a[k] + b[k]) / 2 for k in M}
    spread = {k: abs(a[k] - b[k]) / 2 for k in M}
    secs_mean = (a["inference_s"] + b["inference_s"]) / 2
    rows = [("Dataset", "QuranMB.v1", f"QuranMB.v1 = full QuranMB.v2 audio ({a['clips']} clips), {v1lab}"),
            ("PER / P / R / F1 (%)", " / ".join(f"{p[k]:.2f}" for k in M),
             " / ".join(f"{mean[k]:.2f} ± {spread[k]:.2f}" for k in M)),
            ("Training", "12.5k updates, batch 16", f"not retrained: released checkpoint (stored Step {CKINFO[name]['step']})"),
            ("Inference time", "not reported", f"{secs_mean / 60:.1f} min per pass on {ENV['inference_env']['device']} "
                                                f"({secs_mean / (secs / 3600):.0f} s per audio hour)")]
    print(f"\n=== {name} ===")
    print(pd.DataFrame(rows, columns=["Evaluation item", "Original paper", "Our experiment"]).to_string(index=False))
    print("difference (ours mean - paper): " + ", ".join(f"{k} {mean[k] - p[k]:+.2f}" for k in M))
    SUMMARY.append({"model": name, **{f"paper_{k}": p[k] for k in M}, **{f"mean_{k}": round(mean[k], 2) for k in M},
                    **{f"spread_{k}": round(spread[k], 2) for k in M}, **{f"eval_{k}": e[k] for k in M}})

print("\nEXTRA (labelled): released script + .eval(), deterministic (dropout off, as the s3prl runner evaluates)")
ev = pd.DataFrame([{"model": n, **{k: row(n, "eval")[k] for k in M},
                    **{f"{k} - paper": round(row(n, "eval")[k] - PAPER[n][i], 2) for i, k in enumerate(M)}} for n in MODELS])
print(ev.to_string(index=False))
pd.DataFrame(SUMMARY).to_csv(OUT / "comparison_summary.csv", index=False)
if SETS["v1"] == SETS["v2"]:
    mh = row("mhubert", "eval")
    print("\nFull QuranMB.v2 with the current labels: the same labels as above (no older answer file exists), so the tables "
          "above are also the v2 tables. Lab-project baseline = mHuBERT in eval mode (deterministic; our own models were "
          f"scored in eval mode): F1 {mh['F1']:.2f}, P {mh['P']:.2f}, R {mh['R']:.2f}, PER {mh['PER']:.2f}")
else:
    print("\n=== Extra: full QuranMB.v2 with the current labels ===")
    print(results[results["labels"] == SETS["v2"]][["model", "variant"] + M].to_string(index=False))

def fmt(name, variant):
    r = row(name, variant)
    return ", ".join(f"{k} {r[k]:.2f}" for k in M)
deviation("dropout at inference", "the released s3prl_inference.py never calls .eval(): the upstream, featurizer and "
          "downstream were all in training mode at inference (recorded per run in inference_modes.json), so dropout (0.2 in "
          "the BiLSTM per config/sws.yaml, plus the upstream's) was on and predictions vary between runs "
          f"(clips that changed between seeds 1 and 2: {changed}). The s3prl runner's evaluation path calls .eval(). "
          "Main table: the released script as published, mean ± spread over seeds 1 and 2. Labelled extra: the same script "
          "with .eval() (deterministic). Evidence: " + "; ".join(
              f"{n}: seed 1 [{fmt(n, 'released_seed1')}], seed 2 [{fmt(n, 'released_seed2')}], eval [{fmt(n, 'eval')}]"
              for n in MODELS))


In [ ]:
# ---- S6. Evidence and the Deviations cell ------------------------------------------------------------------------------
ENV["truth"] = TRUTH_INFO
ENV["inference_times"] = {f"{k[0]}/{k[1]}": v for k, v in TIMES.items()}
ENV["checkpoints"] = CKINFO
(OUT / "environment.json").write_text(json.dumps(ENV, indent=1, default=str))
(OUT / "deviations.json").write_text(json.dumps(DEVIATIONS, indent=1))
print("DEVIATIONS")
for d in DEVIATIONS:
    print(f"{d['n']}. [{d['topic']}] {d['deviation']}\n")
print("saved to", OUT, ":", sorted(p.name for p in OUT.iterdir()))
print("Next: Save Version, then Output -> New Dataset named 'a2-reproduction-outputs'.")
